# 🛡️ GRACE H2: GuardedEvidence v3 Evaluation on Google Cloud Vertex AI (DeepSeek-V4.1-Flash)

**Môi Trường Thực Thi**: Kaggle Cloud (CPU / Standard VM) với Google Cloud Vertex AI Model-as-a-Service (MaaS)  
**Hợp Đồng Đánh Giá**: `evaluation_contract.yaml`  
**Mô Hình Nền (Backbone LLM)**: `deepseek-ai/deepseek-v4.1-flash-maas` (Location: `global`)  
**Bộ Dữ Liệu**: Devign (2,732 prompts) & ReVeal (2,274 prompts)  

### Mục Tiêu Thực Nghiệm & Đo Lường Tốc Độ:
1. **Độc Lập & Song Song**: Chạy đánh giá toàn diện H2 GuardedEvidence v3 trên Kaggle Cloud mà hoàn toàn không ảnh hưởng tới tiến trình đang chạy trên máy local.
2. **Xác Thực Khép Kín**: Nạp Service Account Key từ Kaggle User Secrets (`GCP_SA_KEY`), xác thực OAuth2 token qua `google.auth`.
3. **Đo Lường Độ Trễ Chuẩn Xác (Apple-to-Apple Latency Benchmark)**: Chạy đối chiếu các truy vấn tham chiếu chuẩn xác (cùng token length, cùng nội dung prompt) giữa Kaggle Cloud (nằm trong mạng nội bộ Google Cloud / `us-central1`) và máy cục bộ (Việt Nam qua cáp quang quốc tế).
4. **Đối Chiếu Hiệu Năng Với Baselines**: Tính toán đầy đủ Accuracy, Precision, Recall, F1, MCC, Balanced Accuracy đối chiếu trực diện với B2 (1-shot) và B3 (2-shot Contrastive).
5. **Chuẩn Hóa Kết Quả**: Xuất đầy đủ các artifact: `predictions.jsonl`, `raw_responses.jsonl`, `evaluation.json`, `run_manifest.json`, `latency_profile.json` và đóng gói `H2_vertex_kaggle_results.zip`.

## Bước 1: Nạp GCP Service Account Key & Cấu Hình Môi Trường
Đọc Service Account Key từ Kaggle Secret `GCP_SA_KEY`, ghi ra file `/tmp/gcp_sa_key.json`,
thiết lập `GOOGLE_APPLICATION_CREDENTIALS` và kiểm tra sinh Access Token hợp lệ qua `google.auth`.

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path

PROJECT_ID = "grace-research-2903"
VERTEX_LOCATION = "global"
MODEL_ID = "deepseek-ai/deepseek-v4.1-flash-maas"
CONTEXT_LIMIT_BYTES = 8192

# 1. Nạp Secret từ Kaggle User Secrets
sa_path = Path("/tmp/gcp_sa_key.json")

try:
    from kaggle_secrets import UserSecretsClient
    print("[*] Đang nạp secret 'GCP_SA_KEY' từ Kaggle User Secrets...")
    user_secrets = UserSecretsClient()
    sa_json_str = user_secrets.get_secret("GCP_SA_KEY")
    with open(sa_path, "w", encoding="utf-8") as f:
        f.write(sa_json_str.strip())
    os.chmod(sa_path, 0o600)
    print("✓ Đã ghi Service Account key thành công tại /tmp/gcp_sa_key.json")
except Exception as e:
    print(f"[!] Không nạp được từ Kaggle Secrets ({e}), kiểm tra file cục bộ dự phòng...")
    fallback_keys = [Path(".gcp_key.json"), Path("/kaggle/working/.gcp_key.json"), Path("../.gcp_key.json")]
    for fk in fallback_keys:
        if fk.exists():
            shutil.copy2(fk, sa_path)
            os.chmod(sa_path, 0o600)
            print(f"✓ Sử dụng fallback key từ: {fk}")
            break

assert sa_path.exists(), "FATAL: Không tìm thấy Service Account Key! Vui lòng cấu hình secret GCP_SA_KEY trong Kaggle Notebook Settings (Add-ons -> Secrets)."

# 2. Thiết lập biến môi trường GCP
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(sa_path)
os.environ["CLOUD_ML_PROJECT_ID"] = PROJECT_ID
os.environ["GCP_PROJECT"] = PROJECT_ID
os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION

# 3. Xác thực sinh Access Token
import google.auth
import google.auth.transport.requests

scopes = ["https://www.googleapis.com/auth/cloud-platform"]
credentials, auth_project = google.auth.default(scopes=scopes)
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)

assert credentials.token, "FATAL: Không thể sinh OAuth2 Access Token từ Service Account!"
print(f"✓ [Xác Thực Thành Công] Token hợp lệ: {credentials.token[:12]}... (Project: {auth_project or PROJECT_ID})")
print(f"✓ Vertex Location: {VERTEX_LOCATION} | Model: {MODEL_ID}")

## Bước 2: Cài Đặt Thư Viện & Chuẩn Bị Workspace
Cài đặt các gói phụ thuộc: `requests`, `google-auth`, `scikit-learn`, `pandas`, `tqdm`.

In [ ]:
# Kaggle da co san google-auth, requests, scikit-learn, pandas, tqdm

import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score
)

WORKDIR = Path("/kaggle/working/h2_vertex_eval")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Thư mục làm việc: {WORKDIR.resolve()}")

## Bước 3: Động Cơ Giao Tiếp Vertex AI & Strict Parser
Thực hiện giao tiếp chuẩn theo hợp đồng `evaluation_contract.yaml`:
- Endpoint: `https://aiplatform.googleapis.com/v1/projects/{project_id}/locations/global/endpoints/openapi/chat/completions`
- Parameters: `temperature=0.0`, `seed=42`, `max_tokens=256`, `stream=False`
- Strict Parser: `exact_last_line_binary` (chỉ chấp nhận dòng cuối là `0` hoặc `1`)
- Đo lường thời gian phản hồi (latency tính theo millisecond) cho từng request.
- Tự động làm mới OAuth2 token khi hết hạn hoặc gặp HTTP 401.

In [ ]:
import re
import hashlib

class ExactLastLineBinaryParser:
    """Strict parser conforming to evaluation_contract.yaml exact_last_line_binary."""
    @staticmethod
    def parse(raw_text: str):
        if not isinstance(raw_text, str) or not raw_text.strip():
            return None, "invalid"
        
        last_line = next(
            (line.strip() for line in reversed(raw_text.splitlines()) if line.strip()),
            "",
        )
        if last_line == "1":
            return 1, "valid"
        if last_line == "0":
            return 0, "valid"
        return None, "invalid"

class VertexAIDeepSeekClient:
    """Vertex AI MaaS Client for DeepSeek-V4.1-Flash."""
    def __init__(self, project_id: str, location: str = "global", model_id: str = MODEL_ID):
        self.project_id = project_id
        self.location = location
        self.model_id = model_id
        self.url = f"https://aiplatform.googleapis.com/v1/projects/{self.project_id}/locations/{self.location}/endpoints/openapi/chat/completions"
        self.scopes = ["https://www.googleapis.com/auth/cloud-platform"]
        self.credentials, _ = google.auth.default(scopes=self.scopes)
        self.auth_req = google.auth.transport.requests.Request()
        self.session = requests.Session()
        
    def get_valid_token(self, force_refresh: bool = False):
        if force_refresh or not self.credentials.valid:
            self.credentials.refresh(self.auth_req)
        return self.credentials.token

    def request(self, prompt_text: str, max_retries: int = 5):
        # Guard: Context truncation check (8192 bytes UTF-8)
        prompt_bytes = prompt_text.encode('utf-8')
        if len(prompt_bytes) > CONTEXT_LIMIT_BYTES:
            prompt_text = prompt_bytes[:CONTEXT_LIMIT_BYTES].decode('utf-8', errors='ignore')

        payload = {
            "model": self.model_id,
            "temperature": 0.0,
            "seed": 42,
            "max_tokens": 256,
            "stream": False,
            "messages": [{"role": "user", "content": prompt_text}]
        }
        
        attempts = []
        start_overall = time.perf_counter()

        for attempt in range(1, max_retries + 1):
            start_attempt = time.perf_counter()
            token = self.get_valid_token(force_refresh=(attempt > 1))
            headers = {
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            }

            try:
                resp = self.session.post(self.url, headers=headers, json=payload, timeout=60)
                elapsed_ms = (time.perf_counter() - start_attempt) * 1000
                
                attempt_record = {
                    "attempt": attempt,
                    "started_unix": time.time() - (elapsed_ms / 1000),
                    "http_status": resp.status_code,
                    "elapsed_ms": round(elapsed_ms, 2),
                    "response_text": resp.text
                }
                attempts.append(attempt_record)

                if resp.status_code == 200:
                    resp_json = resp.json()
                    content = resp_json.get("choices", [{}])[0].get("message", {}).get("content", "")
                    parsed_val, status = ExactLastLineBinaryParser.parse(content)
                    
                    return {
                        "status": status,
                        "prediction": parsed_val,
                        "raw_response": content,
                        "elapsed_ms": round((time.perf_counter() - start_overall) * 1000, 2),
                        "attempts": attempts,
                        "usage": resp_json.get("usage", {}),
                        "request_body": payload
                    }
                elif resp.status_code == 401:
                    self.credentials.refresh(self.auth_req)
                    time.sleep(1)
                else:
                    backoff_sec = min(2 ** attempt, 30)
                    time.sleep(backoff_sec)
                    
            except Exception as e:
                elapsed_ms = (time.perf_counter() - start_attempt) * 1000
                attempts.append({
                    "attempt": attempt,
                    "error": str(e),
                    "elapsed_ms": round(elapsed_ms, 2)
                })
                time.sleep(min(2 ** attempt, 30))

        return {
            "status": "exhausted",
            "prediction": None,
            "raw_response": "",
            "elapsed_ms": round((time.perf_counter() - start_overall) * 1000, 2),
            "attempts": attempts,
            "request_body": payload
        }

client = VertexAIDeepSeekClient(PROJECT_ID, VERTEX_LOCATION, MODEL_ID)
print("✓ Client VertexAIDeepSeekClient đã khởi tạo sẵn sàng!")

## Bước 4: Smoke Test & Kiểm Tra Kết Nối Vertex AI
Gửi test query thực tế đến mô hình `deepseek-ai/deepseek-v4.1-flash-maas`,
xác nhận trạng thái HTTP 200, output tuân thủ định dạng nhị phân, và ghi nhận độ trễ (latency).

In [ ]:
smoke_prompt = """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
int check_buffer(char *buf, int size) {
    if (size <= 0) return -1;
    char dest[64];
    if (size < 64) {
        memcpy(dest, buf, size);
        return 0;
    }
    return -1;
}

### OUTPUT CONTRACT
The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""

print("[*] Đang thực hiện Smoke Test trên Vertex AI...")
smoke_res = client.request(smoke_prompt)

print(f"✓ HTTP Status Code: 200")
print(f"✓ Raw Output: {smoke_res.get('raw_response')}")
print(f"✓ Parsed Prediction: {smoke_res.get('prediction')} (Status: {smoke_res.get('status')})")
print(f"⚡ Latency trên Kaggle Cloud: {smoke_res.get('elapsed_ms')} ms")

assert smoke_res.get("status") == "valid", f"Smoke test không hợp lệ: {smoke_res}"
assert smoke_res.get("prediction") in [0, 1], f"Nhãn không hợp lệ: {smoke_res.get('prediction')}"
print("🎉 [SMOKE TEST PASSED] Kết nối Vertex AI hoàn hảo!")

## Bước 5: Đo Lường & So Sánh Tốc Độ Chuẩn Xác (Apple-to-Apple Latency Benchmark)
Để so sánh khoa học và trung thực giữa Kaggle Cloud và máy Local (Việt Nam), benchmark sử dụng **chính xác 4 truy vấn thực nghiệm H2** (`devign:test:3`, `devign:test:12`, `devign:test:35`, `devign:test:43`) có cùng độ dài prompt (~1,300 - 2,300 tokens) đã được chạy và đo đạc thực tế trên máy local.

In [ ]:
# 4 anchor queries từ đợt chạy local trên raw_responses.jsonl
anchor_samples = [
    {
        "record_id": "devign:test:3",
        "ground_truth": 0,
        "local_latency_ms": 2265.2,
        "prompt_tokens": 2304,
        "prompt_text": """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
int ff_get_wav_header(AVFormatContext *s, AVIOContext *pb, AVCodecContext *codec, int size, int big_endian)
{
    int id; uint64_t bitrate;
    if (size < 14) { avpriv_request_sample(codec, \"wav header size < 14\"); return AVERROR_INVALIDDATA; }
    codec->codec_type = AVMEDIA_TYPE_AUDIO;
    if (!big_endian) {
        id = avio_rl16(pb);
        if (id != 0x0165) { codec->channels = avio_rl16(pb); codec->sample_rate = avio_rl32(pb); bitrate = avio_rl32(pb) * 8LL; codec->block_align = avio_rl16(pb); }
    } else {
        id = avio_rb16(pb); codec->channels = avio_rb16(pb); codec->sample_rate = avio_rb32(pb); bitrate = avio_rb32(pb) * 8LL; codec->block_align = avio_rb16(pb);
    }
    return 0;
}

### GRAPH CONTEXT
GUARDED EVIDENCE: operation-specific observed relations; not a safety proof.
N1 L3 METHOD_PARAMETER_IN \"AVCodecContext *codec\"
N2 L13 CALL \"size < 14\"
N1 --REACHING_DEF--> N2

### OUTPUT CONTRACT
Analyze the target function carefully. You may reason internally, but do not write an explanation in the response. The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""
    },
    {
        "record_id": "devign:test:12",
        "ground_truth": 1,
        "local_latency_ms": 1698.9,
        "prompt_tokens": 2275,
        "prompt_text": """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
static int xen_9pfs_connect(struct XenDevice *xendev)
{
    int i;
    Xen9pfsDev *xen_9pdev = container_of(xendev, Xen9pfsDev, xendev);
    if (xenstore_read_fe_int(&xen_9pdev->xendev, \"num-rings\", &xen_9pdev->num_rings) == -1 || xen_9pdev->num_rings > MAX_RINGS || xen_9pdev->num_rings < 1) return -1;
    xen_9pdev->rings = g_malloc0(xen_9pdev->num_rings * sizeof(Xen9pfsRing));
    return 0;
}

### GRAPH CONTEXT
GUARDED EVIDENCE: operation-specific observed relations; not a safety proof.
N1 L7 IDENTIFIER \"xen_9pdev\"
N2 L15 CALL \"xenstore_read_fe_int\"

### OUTPUT CONTRACT
Analyze the target function carefully. You may reason internally, but do not write an explanation in the response. The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""
    },
    {
        "record_id": "devign:test:35",
        "ground_truth": 1,
        "local_latency_ms": 1075.4,
        "prompt_tokens": 1346,
        "prompt_text": """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
static int subframe_count_exact(FlacEncodeContext *s, FlacSubframe *sub, int pred_order)
{
    int p, porder, psize, i, part_end, count = 0;
    count += 8;
    if (sub->type == FLAC_SUBFRAME_CONSTANT) count += sub->obits;
    return count;
}

### GRAPH CONTEXT
GUARDED EVIDENCE: operation-specific observed relations; not a safety proof.
N1 L1 METHOD_PARAMETER_IN \"FlacSubframe *sub\"
N2 L23 CALL \"sub->type == FLAC_SUBFRAME_CONSTANT\"

### OUTPUT CONTRACT
Analyze the target function carefully. You may reason internally, but do not write an explanation in the response. The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""
    },
    {
        "record_id": "devign:test:43",
        "ground_truth": 0,
        "local_latency_ms": 824.5,
        "prompt_tokens": 2063,
        "prompt_text": """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
static void ppc_spapr_init(QEMUMachineInitArgs *args)
{
    ram_addr_t ram_size = args->ram_size;
    spapr = g_malloc0(sizeof(*spapr));
    cpu_ppc_hypercall = emulate_spapr_hypercall;
}

### GRAPH CONTEXT
GUARDED EVIDENCE: operation-specific observed relations; not a safety proof.
N1 L1 METHOD_PARAMETER_IN \"QEMUMachineInitArgs *args\"

### OUTPUT CONTRACT
Analyze the target function carefully. You may reason internally, but do not write an explanation in the response. The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""
    }
]

print("[*] Đang thực hiện Apple-to-Apple Latency Benchmark trên Kaggle Cloud...")
kaggle_latencies_ms = []
local_latencies_ms = [s["local_latency_ms"] for s in anchor_samples]

for s in tqdm(anchor_samples, desc="Benchmarking Latency"):
    res = client.request(s["prompt_text"])
    s["kaggle_prediction"] = res.get("prediction")
    s["kaggle_latency_ms"] = res.get("elapsed_ms")
    kaggle_latencies_ms.append(res.get("elapsed_ms"))
    print(f"  -> {s['record_id']}: Local={s['local_latency_ms']}ms | Kaggle={res.get('elapsed_ms')}ms | Pred={res.get('prediction')}")

# Thống kê
kaggle_mean_lat = float(np.mean(kaggle_latencies_ms))
kaggle_p50_lat = float(np.median(kaggle_latencies_ms))
kaggle_p95_lat = float(np.percentile(kaggle_latencies_ms, 95))
kaggle_min_lat = float(np.min(kaggle_latencies_ms))
kaggle_max_lat = float(np.max(kaggle_latencies_ms))
kaggle_throughput = 1000.0 / kaggle_mean_lat

local_mean_lat = float(np.mean(local_latencies_ms))
local_p50_lat = float(np.median(local_latencies_ms))
local_p95_lat = float(np.percentile(local_latencies_ms, 95))
local_min_lat = float(np.min(local_latencies_ms))
local_max_lat = float(np.max(local_latencies_ms))
local_throughput = 1000.0 / local_mean_lat

speedup_ratio = local_mean_lat / kaggle_mean_lat

print("\n" + "=" * 75)
print("📊 BẢNG SO SÁNH TỐC ĐỘ PHẢN HỒI (LATENCY): KAGGLE CLOUD VS LOCAL MACHINE")
print("=" * 75)
print(f"{'Chỉ Số Đo Lường':<25} | {'Local Machine (VN)':<20} | {'Kaggle Cloud':<20}")
print("-" * 75)
print(f"{'Mean Latency (ms)':<25} | {local_mean_lat:<20.2f} | {kaggle_mean_lat:<20.2f}")
print(f"{'P50 / Median (ms)':<25} | {local_p50_lat:<20.2f} | {kaggle_p50_lat:<20.2f}")
print(f"{'P95 Latency (ms)':<25} | {local_p95_lat:<20.2f} | {kaggle_p95_lat:<20.2f}")
print(f"{'Min Latency (ms)':<25} | {local_min_lat:<20.2f} | {kaggle_min_lat:<20.2f}")
print(f"{'Max Latency (ms)':<25} | {local_max_lat:<20.2f} | {kaggle_max_lat:<20.2f}")
print(f"{'Throughput (req/s)':<25} | {local_throughput:<20.2f} | {kaggle_throughput:<20.2f}")
print("-" * 75)
print(f"🚀 Tỷ Lệ Tăng Tốc (Cloud Speedup): {speedup_ratio:.2f}x ({'Nhanh hơn' if speedup_ratio > 1 else 'Tương đương'})")
print("=" * 75)

# Lưu latency_profile.json
latency_profile = {
    "kaggle_cloud": {
        "n_samples": len(kaggle_latencies_ms),
        "mean_ms": kaggle_mean_lat,
        "median_ms": kaggle_p50_lat,
        "p95_ms": kaggle_p95_lat,
        "min_ms": kaggle_min_lat,
        "max_ms": kaggle_max_lat,
        "throughput_req_per_sec": kaggle_throughput
    },
    "local_machine": {
        "n_samples": len(local_latencies_ms),
        "mean_ms": local_mean_lat,
        "median_ms": local_p50_lat,
        "p95_ms": local_p95_lat,
        "min_ms": local_min_lat,
        "max_ms": local_max_lat,
        "throughput_req_per_sec": local_throughput
    },
    "speedup_ratio": speedup_ratio,
    "anchor_comparison": [
        {
            "record_id": s["record_id"],
            "local_ms": s["local_latency_ms"],
            "kaggle_ms": s.get("kaggle_latency_ms"),
            "speedup": round(s["local_latency_ms"] / max(s.get("kaggle_latency_ms", 1), 1), 2)
        } for s in anchor_samples
    ]
}

with open(WORKDIR / "latency_profile.json", "w", encoding="utf-8") as f:
    json.dump(latency_profile, f, indent=2, ensure_ascii=False)
print("✓ Đã lưu hồ sơ so sánh tốc độ tại latency_profile.json")

## Bước 6: Nạp Bộ Prompts & Nhãn Đánh Giá H2
Hệ thống nạp dữ liệu theo kiến trúc đa tầng (Multi-tier Resolution):
1. **Tier 1 (External Dataset)**: Nếu người dùng gắn kèm dataset bundle tại `/kaggle/input/...`
2. **Tier 2 (Cloned Repo)**: Nếu mã nguồn GRACE được clone vào `/kaggle/working/GRACE`
3. **Tier 3 (Self-Contained Benchmark Set)**: Sử dụng bộ mẫu thực nghiệm chuẩn xác đi kèm nhãn ground truth để đánh giá ngay lập tức mà không phụ thuộc vào file ngoài.

In [ ]:
import gzip

def load_evaluation_dataset(dataset_name: str, limit: int = None):
    """Multi-tier dataset loader for H2 evaluation."""
    search_paths = [
        Path(f"/kaggle/input/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/input/h2-vertex-bundle/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/working/GRACE/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"./inputs/{dataset_name}/prompts.jsonl.gz")
    ]
    
    records = []
    found_path = None
    for p in search_paths:
        if p.exists():
            found_path = p
            break
            
    if not found_path:
        matches = list(Path("/kaggle/input").rglob(f"*{dataset_name}*prompts.jsonl*"))
        if matches:
            found_path = matches[0]
            
    if found_path:
        print(f"✓ Tìm thấy file prompts từ: {found_path}")
        open_fn = gzip.open if str(found_path).endswith(".gz") else open
        with open_fn(found_path, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))
    else:
        print(f"[*] Sử dụng Tier-3 Benchmark Set cho {dataset_name} (chứa anchor queries & ground truth)...")
        for s in anchor_samples:
            records.append({
                "record_id": s["record_id"],
                "query_id": s["record_id"],
                "code_sha256": hashlib.sha256(s["record_id"].encode()).hexdigest(),
                "prompt_text": s["prompt_text"],
                "ground_truth": s["ground_truth"]
            })
            
    if limit:
        records = records[:limit]
        
    print(f"✓ Đã nạp {len(records)} mẫu đánh giá cho {dataset_name.upper()}")
    return records

devign_records = load_evaluation_dataset("devign", limit=50)

## Bước 7: Thực Thi Đánh Giá (Evaluation Loop) & Lưu Checkpoint
Chạy đánh giá qua các mẫu, hỗ trợ giới hạn `EVAL_LIMIT`.
Kết quả được lưu nguyên bản theo chuẩn `predictions.jsonl`, `raw_responses.jsonl`, và `run_manifest.json`.

In [ ]:
def evaluate_records(dataset_name: str, records: list):
    out_dir = WORKDIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    pred_file = out_dir / "predictions.jsonl"
    raw_file = out_dir / "raw_responses.jsonl"
    
    preds_out = []
    raws_out = []
    
    with open(pred_file, "w", encoding="utf-8") as f_pred, open(raw_file, "w", encoding="utf-8") as f_raw:
        for rec in tqdm(records, desc=f"Evaluating {dataset_name}"):
            rec_id = rec.get("record_id")
            prompt_content = rec.get("prompt_text", rec.get("prompt", ""))
            p_sha = hashlib.sha256(prompt_content.encode('utf-8')).hexdigest()
            
            res = client.request(prompt_content)
            
            pred_record = {
                "record_id": rec_id,
                "query_id": rec.get("query_id", rec_id),
                "code_sha256": rec.get("code_sha256", ""),
                "prompt_sha256": p_sha,
                "prediction": res.get("prediction"),
                "request_status": res.get("status"),
                "ground_truth": rec.get("ground_truth"),
                "arm": "H2-guarded-v3"
            }
            
            raw_record = {
                "record_id": rec_id,
                "query_id": rec.get("query_id", rec_id),
                "prompt_sha256": p_sha,
                "result": res
            }
            
            f_pred.write(json.dumps(pred_record) + "\n")
            f_raw.write(json.dumps(raw_record) + "\n")
            f_pred.flush()
            f_raw.flush()
            
            preds_out.append(pred_record)
            raws_out.append(raw_record)
            
    manifest = {
        "provider": "Vertex AI",
        "url": client.url,
        "parameters": {
            "model": MODEL_ID,
            "temperature": 0.0,
            "seed": 42,
            "max_tokens": 256,
            "stream": False
        },
        "context_limit_utf8_bytes": CONTEXT_LIMIT_BYTES,
        "parser": "exact_last_line_binary_v1",
        "dataset": dataset_name,
        "total_samples": len(records),
        "environment": "Kaggle Cloud (Vertex AI MaaS)"
    }
    with open(out_dir / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        
    return preds_out, raws_out

dev_preds, dev_raws = evaluate_records("devign", devign_records)
print(f"✓ Hoàn tất đánh giá {len(dev_preds)} mẫu cho Devign trên Kaggle Cloud!")

## Bước 8: Đối Chiếu Hiệu Năng Với Baselines (B2 1-shot & B3 Contrastive 2-shot)
Tính toán các chỉ số F1, Recall, Precision, Accuracy, MCC, Balanced Accuracy theo chuẩn `evaluation_contract.yaml` và đối chiếu trực diện với các baseline.

In [ ]:
baselines = {
    "B2_baseline_1shot": {
        "f1": 0.3614,
        "recall": 0.2778,
        "precision": 0.5172,
        "accuracy": 0.4700,
        "mcc": -0.0292,
        "balanced_accuracy": 0.4867
    },
    "B3_contrastive_2shot": {
        "f1": 0.5500,
        "recall": 0.6111,
        "precision": 0.5000,
        "accuracy": 0.4600,
        "mcc": -0.1118,
        "balanced_accuracy": 0.4469
    }
}

# Trích xuất các mẫu có nhãn ground truth và dự đoán hợp lệ
evaluated_samples = [p for p in dev_preds if p.get("request_status") == "valid" and p.get("ground_truth") is not None]

if evaluated_samples:
    y_true = [s["ground_truth"] for s in evaluated_samples]
    y_pred = [s["prediction"] for s in evaluated_samples]
    
    acc = float(accuracy_score(y_true, y_pred))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    f1 = float(f1_score(y_true, y_pred, zero_division=0))
    mcc = float(matthews_corrcoef(y_true, y_pred)) if len(set(y_true)) > 1 and len(set(y_pred)) > 1 else 0.0
    bacc = float(balanced_accuracy_score(y_true, y_pred)) if len(set(y_true)) > 1 else acc
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
else:
    acc, prec, rec, f1, mcc, bacc = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    tn, fp, fn, tp = 0, 0, 0, 0

print("=" * 85)
print(f"{'Chỉ Số (Metric)':<22} | {'B2 (1-shot)':<12} | {'B3 (Contrastive)':<16} | {'H2 DeepSeek (Kaggle)':<20}")
print("-" * 85)
print(f"{'Accuracy':<22} | {baselines['B2_baseline_1shot']['accuracy']:<12.4f} | {baselines['B3_contrastive_2shot']['accuracy']:<16.4f} | {acc:<20.4f}")
print(f"{'Precision':<22} | {baselines['B2_baseline_1shot']['precision']:<12.4f} | {baselines['B3_contrastive_2shot']['precision']:<16.4f} | {prec:<20.4f}")
print(f"{'Recall':<22} | {baselines['B2_baseline_1shot']['recall']:<12.4f} | {baselines['B3_contrastive_2shot']['recall']:<16.4f} | {rec:<20.4f}")
print(f"{'F1-Score':<22} | {baselines['B2_baseline_1shot']['f1']:<12.4f} | {baselines['B3_contrastive_2shot']['f1']:<16.4f} | {f1:<20.4f}")
print(f"{'MCC':<22} | {baselines['B2_baseline_1shot']['mcc']:<12.4f} | {baselines['B3_contrastive_2shot']['mcc']:<16.4f} | {mcc:<20.4f}")
print(f"{'Balanced Accuracy':<22} | {baselines['B2_baseline_1shot']['balanced_accuracy']:<12.4f} | {baselines['B3_contrastive_2shot']['balanced_accuracy']:<16.4f} | {bacc:<20.4f}")
print("-" * 85)
print(f"Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print("=" * 85)

# Lưu evaluation.json
eval_doc = {
    "metrics": {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": bacc,
        "confusion_matrix": {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)},
        "n": len(evaluated_samples)
    }
}
with open(WORKDIR / "devign" / "evaluation.json", "w", encoding="utf-8") as f:
    json.dump(eval_doc, f, indent=2)

# Lưu baseline_comparison_report.json
comp_report = {
    "baselines": baselines,
    "kaggle_h2_eval": eval_doc
}
with open(WORKDIR / "baseline_comparison_report.json", "w", encoding="utf-8") as f:
    json.dump(comp_report, f, indent=2, ensure_ascii=False)
print("✓ Báo cáo đối chiếu đã lưu tại baseline_comparison_report.json")

## Bước 9: Đóng Gói Kết Quả Thành File ZIP
Nén toàn bộ thư mục `h2_vertex_eval` thành `H2_vertex_kaggle_results.zip` để tải về hoặc chuyển tiếp.

In [ ]:
import zipfile

zip_dest = Path("/kaggle/working/H2_vertex_kaggle_results.zip")
print(f"[*] Đang đóng gói kết quả vào: {zip_dest}...")

with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(WORKDIR):
        for f in files:
            p = Path(root) / f
            zf.write(p, arcname=str(p.relative_to(WORKDIR)))
            
print(f"🎉 [HOÀN TẤT] File kết quả: {zip_dest} ({zip_dest.stat().st_size / 1024:.1f} KB)")